# AMADEUS: Adaptive Music & Audio Diffusion Engine
### Accelerated Google Colab GPU Full Training & Inference Suite

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Active GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

## 2. Install Required Dependencies & Configure Path

In [ ]:
!pip install -q torchaudio soundfile librosa transformers accelerate tqdm fastapi uvicorn pydantic pyyaml websockets
!pip install -e /content/AMADEUS 2>/dev/null || !pip install -e .
import sys
import os
for p in ['/content/AMADEUS', '/content/amadeus', '.']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
if os.path.exists('/content/AMADEUS') and not os.path.exists('/content/amadeus'):
    try:
        os.symlink('/content/AMADEUS', '/content/amadeus')
    except Exception:
        pass
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

## 3. Mount Google Drive for Continuous Checkpointing

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/amadeus_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Persistent checkpoint directory set to:', CHECKPOINT_DIR)

## 4. Download Real Music Dataset (GTZAN or FMA)

In [ ]:
from amadeus.data.dataset import download_real_music_dataset, MusicAudioDataset
DATASET_DIR = './real_music_dataset'
download_real_music_dataset(DATASET_DIR, dataset_name='gtzan')
ds = MusicAudioDataset(DATASET_DIR, duration_seconds=6.0)
print('Real music dataset loaded with sample count:', len(ds))

## 5. Turnkey Full Model Training (All Stages with Resuming)

In [ ]:
!python scripts/train_full.py \
    --data_dir ./real_music_dataset \
    --dataset custom \
    --checkpoints_dir /content/drive/MyDrive/amadeus_checkpoints \
    --stage all \
    --epochs_codec 10 \
    --epochs_dit 10 \
    --epochs_lm 10 \
    --batch_size 4 \
    --grad_accum 2 \
    --device cuda \
    --resume

## 6. Run Evaluation Suite & Metrics

In [ ]:
from amadeus.pipeline.inference import AMADEUSPipeline
from amadeus.eval.metrics import EvaluationSuite

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipeline = AMADEUSPipeline(device=device, checkpoint_dir='/content/drive/MyDrive/amadeus_checkpoints')
suite = EvaluationSuite()
metrics = suite.run_full_evaluation(pipeline, test_samples=3)
print('Mean Latency (ms):', metrics['mean_latency_ms'])
print('Mean STFT Distance:', metrics['mean_stft_distance'])
print('Mean Style Cosine:', metrics['mean_style_cosine'])
print('FAD Score:', metrics['fad_score'])
print('Beat F1:', metrics['mean_beat_f1'])
print('Targets Status:', metrics['targets_met'])

## 7. Interactive Generation & In-Browser Audio Playback

In [ ]:
import IPython.display as ipd

prompt = 'cyberpunk synthwave with driving bass and drums, futuristic city sunset'
print('Generating soundtrack for prompt:', prompt)
result = pipeline.generate(
    text_prompt=prompt,
    controls={'steps': 8, 'duration_s': 10, 'stems': True}
)

audio_np = result['audio'].cpu().numpy()
print('Job ID:', result['job_id'])
print('Watermark ID:', result['watermark_id'])
print('Key:', result['analysis']['key'], result['analysis']['mode'], '| BPM:', result['analysis']['bpm'])

ipd.display(ipd.Audio(audio_np, rate=48000))